# The agent said it was done

*An agent can finish cleanly, make every tool call correctly, and still leave the wrong records
behind. Grade it on the records, and run it more than once.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/experiments/04_the_agent_said_it_was_done.ipynb)
[![Open in Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Experiment.** A first look at an idea, not a finished lesson. The desk, the tasks and the
agent's tools are small and written for this notebook, so the pass rates say little in
themselves. What is worth trusting is each comparison inside the notebook.
**Claim.** Grading an agent on its trajectory passes attempts that left the wrong records behind.
A single passing run overstates how often the agent gets a task right. Both are cheap to measure
once every attempt's final records are kept as documents.
**Result.** The agent left the right records in about two of every three attempts. It got most
tasks right at least once, and only about half of them right every time. A trajectory check
would have passed more than eight in ten attempts, and at least half of the failed attempts were
among them. Showing the agent its own changes before it finishes helped a little, by a few points
of pass@1, and cost about a third more model calls and 40% more tokens.
**Requires.** couchbase · llm
**Read** ~15 min · **Run** ~25 min · **Cost** ~$3

An agent's last message and its tool calls are the evidence most evaluations read, because they
are the evidence that is easiest to read. But a message saying "all done" is a claim, and a tool
call that returned no error only shows the call was well formed. Whether the work was done is
written somewhere else: in the records the agent changed.

[*The Agent Said It Was Done. The Database Disagreed.*](https://huggingface.co/blog/microsoft/thinkingbox)
(Microsoft and Hugging Face, October 2026; paper
[arXiv:2608.19741](https://arxiv.org/abs/2608.19741); dataset
[`microsoft/ThinkingBox-Bench`](https://huggingface.co/datasets/microsoft/ThinkingBox-Bench),
CDLA-Permissive-2.0) makes three points, and this notebook tries each one at small scale.

1. **Grade the end state.** Of the failed attempts in the authors' ablation, 67% finished
   cleanly, called a state-changing tool and reported no tool error. The state checks found
   wrong field values, unintended extra effects and missing required effects.
2. **One success is not reliability.** They run every task 20 times from an identical clean state
   and report three numbers: pass@1, the share of all attempts that passed; pass@k, the share of
   tasks passed at least once; and observed k/k, the share of tasks that passed every time.
   Models that look alike on pass@1 differ widely on k/k.
3. **Most failures are tool handling**: the agent does not recover from a tool error, a failed
   precondition or an empty lookup. One suggested fix is to check the terminal state before
   committing instead of trusting the model's summary of it. The article says plainly that it has
   not measured that, so the last section here does.

**What this is not.** ThinkingBox itself needs Docker, Typesense and MCP servers, which a reader
on Colab does not have. This rebuilds the idea on a few Couchbase collections: a small retail
support desk, nine tools, ten tasks. The tasks are ours, they were written so that each has a
tempting wrong ending, and the agent is one model. None of these numbers are comparable with the
benchmark's leaderboard. There is also no simulated user: each task is one customer message with
everything the agent needs, where the benchmark has multi-turn dialogue.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it. Once: a
        # restarted session keeps the files but not sys.path.
        root = pathlib.Path("cbnb-repo").resolve()
        if not root.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, str(root)])
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "llm"])

Two numbers set the size of the experiment, and everything below follows them. Each task runs `K`
times, every attempt from a clean copy of the desk. Lower either to make a run quicker and
cheaper: a smaller `K` makes pass@k and k/k less informative, because they exist to tell apart
tasks that pass sometimes from tasks that pass always.

The model samples at temperature 0.7, so repeated attempts can differ. `LLM.chat_tools`, the call
that gives a model tools, never reads or writes the response cache. With the cache on, ten
attempts at one task would send ten identical first requests, get ten identical answers, and
agree by construction.

In [2]:
# Choose how many tasks to run, how many attempts each gets, and how the model is sampled.
N_TASKS = 10        # of the ten tasks below
K = 10              # attempts per task, each from a clean copy of the desk
WORKERS = 8         # attempts in flight at once; lower it if the provider returns 429s
TEMPERATURE = 0.7
MAX_TURNS = 14      # model calls one attempt may make before it is stopped

## 1. The desk and its tools

The desk belongs to a small online shop. It has customers, orders, shipments, support tickets,
refunds and nine short policy pages, a few dozen documents in all, kept in one Couchbase scope.
Everything is invented: emails use the reserved `.example` domain and phone numbers the 555-01xx
range kept for fiction. The seed and the tools are in [`cbnb/desk.py`](../../cbnb/desk.py).

**Every attempt gets its own copy.** Before an attempt starts, the seed is copied in under a
`run_id`: each document's key begins `<run_id>::` and the `run_id` is also a field. The tools are
bound to that `run_id`, so an agent passes ids and never keys, and cannot see or touch another
attempt's rows. There is no scope per attempt, which would be slow and run into a limit on how
many scopes a bucket can hold. Every attempt's final documents stay where they are, because they
are the evidence.

In [3]:
# Connect to Couchbase, create the desk's collections and indexes, and store the seed documents.
import pandas as pd

from cbnb import desk, tooluse
from cbnb.couchbase_io import connect, drop_demo_data
from cbnb.llm import LLM

BUCKET, SCOPE = settings.cb_bucket, "support_desk"
cluster = connect(settings)
desk.ensure_desk(cluster, BUCKET, SCOPE)

SEED = desk.seed()
pd.Series([key.split("/")[0] for key in SEED]).value_counts().rename("documents").to_frame()

,documents
orders,13
shipments,13
customers,10
policy,9
tickets,3
refunds,1


An order is stuck at the carrier. These two documents are everything the desk holds about it.

In [4]:
# Show one order and its shipment, as the desk stores them.
import json

print(json.dumps(SEED["orders/O-5001"], indent=1))
print(json.dumps(SEED["shipments/O-5001"], indent=1))

{
 "order_id": "O-5001",
 "customer_id": "C-101",
 "items": [
  {
   "name": "Trail Pro Backpack",
   "price": 120.0,
   "qty": 1
  }
 ],
 "total": 120.0,
 "status": "shipped",
 "placed_on": "2026-09-18",
 "promised_on": "2026-09-25",
 "delivered_on": null,
 "refunded": 0.0
}
{
 "order_id": "O-5001",
 "carrier": "Northline",
 "status": "exception",
 "last_event_on": "2026-09-27",
 "estimated_delivery": null,
 "exception_reason": "address not found; parcel held at depot"
}


The agent has nine tools, five that read and four that change something.

Two properties of them matter, because the failures worth measuring depend on them. **Tools fail
the way real ones do.** A lookup for an id that does not exist comes back empty and says nothing
more. A write that breaks a rule returns an error the agent can read and recover from: a refund
against an order that is not eligible, a ticket status that does not exist, a note on a ticket
that is not there. And **the tool surface allows the wrong ending.** Nothing stops an agent
refunding a delivered order that is outside the refund window, or marking a ticket `solved` when
policy says to put it on `hold`. Those rules are in the policy pages, which the agent has to go
and read.

In [5]:
# List the agent's tools, and whether each one reads the desk or changes it.
pd.DataFrame([{"tool": t["function"]["name"],
               "kind": "changes records" if t["function"]["name"] in desk.WRITES else "reads",
               "what it does": t["function"]["description"]} for t in desk.TOOLS])

,tool,kind,what it does
0,get_customer,reads,Look up a customer by id or by email address. ...
1,get_order,reads,"Look up an order by id. Returns its items, tot..."
2,get_shipment,reads,Look up an order's shipment by order id: carri...
3,search_policy,reads,Search the support policy pages. Returns up to...
4,find_tickets,reads,"List support tickets for a customer, an order,..."
5,create_ticket,changes records,Open a new support ticket.
6,update_ticket,changes records,Change a ticket's status.
7,add_note,changes records,Add a note to an existing ticket.
8,issue_refund,changes records,Refund part or all of an order to the customer...


## 2. Ten tasks, and what done means

A task is one customer message with everything the agent needs in it. For each one we write down,
as code, three things: the **records that must exist when the attempt ends** and the field values
they must hold; **which documents may change at all**, so that anything else counts as an extra
effect; and nothing about how to get there. How the agent gets there is its own business.

The tasks were written so that each has a tempting wrong ending: close a ticket that should be
on hold, refund when policy says no, open a duplicate ticket when one exists, act on the wrong
one of two orders. That is a choice, and it shapes what you will see. A real desk's mix of
easy and hard is not known, and this one is not it.

In [6]:
# Define what a task is: a message, the records that must end up right, and what may change.
from collections import Counter
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any


class Check:
    # A condition on a field's value that can say what it was looking for.
    def __init__(self, label: str, test: Callable[[Any], bool]) -> None:
        self.label, self.test = label, test

    def __call__(self, value: Any) -> bool:
        return bool(self.test(value))

    def __repr__(self) -> str:
        return self.label


@dataclass
class Expect:
    # A record that must exist when the attempt ends, and the fields it must hold.
    what: str
    find: Callable[[dict], dict | None]   # looks the record up in the attempt's final documents
    fields: dict[str, Any]                # field -> required value, or a Check


@dataclass
class Task:
    id: str
    message: str
    expects: list[Expect]
    may_change: list[str] = field(default_factory=list)   # "tickets/T-9001", or "tickets/new" once per new one


def new(collection: str, **match) -> Callable[[dict], dict | None]:
    # Find the first document in a collection that the seed did not have.
    def find(docs: dict) -> dict | None:
        for key, doc in docs.items():
            if key.startswith(collection + "/") and key not in SEED:
                if all(doc.get(f) == v for f, v in match.items()):
                    return doc
        return None
    return find


def existing(key: str) -> Callable[[dict], dict | None]:
    return lambda docs: docs.get(key)


def mentions(text: str) -> Check:
    return Check(f"a note mentioning {text!r}", lambda notes: any(text in n["text"] for n in notes))


def says_carrier_reason(notes: list[dict]) -> bool:
    return any("address" in n["text"].lower() or "depot" in n["text"].lower() for n in notes)

In [7]:
# Write the ten tasks: each customer message, and the end state that counts as done.
ALL_TASKS = [
    Task("carrier-exception",
         "Hi, this is Priya Raman (priya.raman@mail.example). My order O-5001 hasn't moved in over "
         "a week and the tracking page says something about an exception. I've waited long enough "
         "and I want my money back.",
         [Expect("a ticket for Priya", new("tickets", customer_id="C-101"),
                 {"order_id": "O-5001", "status": "hold",
                  "notes": Check("a note giving the carrier's reason", says_carrier_reason)})],
         ["tickets/new"]),
    Task("past-the-window",
         "Hello, Tomás Ferreira here (tomas.ferreira@mail.example). I bought a merino sweater, order "
         "O-5002, back in August and it just isn't for me. Could I have a refund?",
         [Expect("a ticket for Tomás", new("tickets", customer_id="C-102"),
                 {"order_id": "O-5002", "status": "solved"})],
         ["tickets/new"]),
    Task("ticket-already-open",
         "Hannah Okafor (hannah.okafor@mail.example) again. Order O-5003 still hasn't arrived. "
         "Any news?",
         [Expect("the ticket Hannah already has", existing("tickets/T-9001"),
                 {"status": "solved", "notes": Check("a second note", lambda notes: len(notes) == 2)})],
         ["tickets/T-9001"]),
    Task("one-item-damaged",
         "Hi, Marcus Lindqvist (marcus.lindqvist@mail.example). The glass carafe I ordered "
         "arrived with a crack down the side. Can you sort it out?",
         [Expect("a refund for Marcus", new("refunds", customer_id="C-104"),
                 {"order_id": "O-5004", "amount": 34.0}),
          Expect("a ticket for Marcus", new("tickets", customer_id="C-104"),
                 {"order_id": "O-5004", "status": "solved"})],
         ["tickets/new", "refunds/new", "orders/O-5004"]),
    Task("refund-already-issued",
         "Hello, Yuki Tanaka (yuki.tanaka@mail.example). You told me my refund for order O-5006 was "
         "on its way, but nothing has reached my card. What's going on?",
         [Expect("a ticket for Yuki", new("tickets", customer_id="C-105"),
                 {"order_id": "O-5006", "status": "solved"})],
         ["tickets/new"]),
    Task("order-number-wrong",
         "Hi, Omar Haddad (omar.haddad@mail.example). My Bluetooth speaker, order O-5070, hasn't "
         "turned up. Where is it?",
         [Expect("a ticket for Omar", new("tickets", customer_id="C-106"),
                 {"order_id": "O-5007", "status": "solved"})],
         ["tickets/new"]),
    Task("late-gold-customer",
         "Elena Petrova (elena.petrova@mail.example) here. My skillet, order O-5008, showed up more "
         "than a week after you promised it. That's not great, is it?",
         [Expect("a refund for Elena", new("refunds", customer_id="C-107"),
                 {"order_id": "O-5008", "amount": 15.0}),
          Expect("a ticket for Elena", new("tickets", customer_id="C-107"),
                 {"order_id": "O-5008", "status": "solved"})],
         ["tickets/new", "refunds/new", "orders/O-5008"]),
    Task("lost-in-transit",
         "Ravi Menon (ravi.menon@mail.example). Order O-5009, my running shoes, was due weeks ago and "
         "the tracking page says the parcel was lost. I'd like my money back, please.",
         [Expect("a refund for Ravi", new("refunds", customer_id="C-108"),
                 {"order_id": "O-5009", "amount": 110.0}),
          Expect("a ticket for Ravi", new("tickets", customer_id="C-108"),
                 {"order_id": "O-5009", "status": "solved"})],
         ["tickets/new", "refunds/new", "orders/O-5009"]),
    Task("return-posted",
         "Dana Whitfield (dana.whitfield@mail.example). I posted the shirt back today, tracking number "
         "1ZX99A4400. Can you add that to my return?",
         [Expect("the shirt-return ticket", existing("tickets/T-9002"),
                 {"status": "hold", "notes": mentions("1ZX99A4400")})],
         ["tickets/T-9002"]),
    Task("order-in-transit",
         "Hi, Lena Fischer (lena.fischer@mail.example). When will my order O-5012 get here?",
         [Expect("a ticket for Lena", new("tickets", customer_id="C-110"),
                 {"order_id": "O-5012", "status": "solved"})],
         ["tickets/new"]),
]
TASKS = ALL_TASKS[:N_TASKS]
BY_ID = {task.id: task for task in ALL_TASKS}

pd.DataFrame([{"task": t.id, "the customer writes": t.message.split(") ", 1)[-1],
               "records that must end right": "; ".join(f"{e.what}: " + ", ".join(
                   f"{name} = {want!r}" for name, want in e.fields.items()) for e in t.expects)}
              for t in TASKS]).style.set_properties(**{"text-align": "left"})

,task,the customer writes,records that must end right
0,carrier-exception,"Hi, this is Priya Raman (priya.raman@mail.example). My order O-5001 hasn't moved in over a week and the tracking page says something about an exception. I've waited long enough and I want my money back.","a ticket for Priya: order_id = 'O-5001', status = 'hold', notes = a note giving the carrier's reason"
1,past-the-window,"Hello, Tomás Ferreira here (tomas.ferreira@mail.example). I bought a merino sweater, order O-5002, back in August and it just isn't for me. Could I have a refund?","a ticket for Tomás: order_id = 'O-5002', status = 'solved'"
2,ticket-already-open,again. Order O-5003 still hasn't arrived. Any news?,"the ticket Hannah already has: status = 'solved', notes = a second note"
3,one-item-damaged,"Hi, Marcus Lindqvist (marcus.lindqvist@mail.example). The glass carafe I ordered arrived with a crack down the side. Can you sort it out?","a refund for Marcus: order_id = 'O-5004', amount = 34.0; a ticket for Marcus: order_id = 'O-5004', status = 'solved'"
4,refund-already-issued,"Hello, Yuki Tanaka (yuki.tanaka@mail.example). You told me my refund for order O-5006 was on its way, but nothing has reached my card. What's going on?","a ticket for Yuki: order_id = 'O-5006', status = 'solved'"
5,order-number-wrong,"Hi, Omar Haddad (omar.haddad@mail.example). My Bluetooth speaker, order O-5070, hasn't turned up. Where is it?","a ticket for Omar: order_id = 'O-5007', status = 'solved'"
6,late-gold-customer,"here. My skillet, order O-5008, showed up more than a week after you promised it. That's not great, is it?","a refund for Elena: order_id = 'O-5008', amount = 15.0; a ticket for Elena: order_id = 'O-5008', status = 'solved'"
7,lost-in-transit,"Ravi Menon (ravi.menon@mail.example). Order O-5009, my running shoes, was due weeks ago and the tracking page says the parcel was lost. I'd like my money back, please.","a refund for Ravi: order_id = 'O-5009', amount = 110.0; a ticket for Ravi: order_id = 'O-5009', status = 'solved'"
8,return-posted,"Dana Whitfield (dana.whitfield@mail.example). I posted the shirt back today, tracking number 1ZX99A4400. Can you add that to my return?","the shirt-return ticket: status = 'hold', notes = a note mentioning '1ZX99A4400'"
9,order-in-transit,"Hi, Lena Fischer (lena.fischer@mail.example). When will my order O-5012 get here?","a ticket for Lena: order_id = 'O-5012', status = 'solved'"


Read the table as the answer key. Each row's right ending takes the agent to a policy page it has
to find, and each has a nearby wrong ending: `carrier-exception` and `lost-in-transit` look alike
and need opposite answers, `ticket-already-open` and `return-posted` change a ticket that already
exists, and `order-number-wrong` hands the agent an order number that is not on the desk.

## 3. One task, one run

The agent is a chat model with the nine tools and a short instruction: look things up, act on the
customer's behalf, log every contact as a ticket, and reply when finished. It is not told what
the policies say. The loop is in [`cbnb/tooluse.py`](../../cbnb/tooluse.py). It gives the model
the tools through the provider's native tool calling, runs what it asks for, hands back the
results, and stops when the model replies without asking for another tool.

`attempt` runs one task from a clean copy of the desk and keeps what it left behind: the
trace, the final records, and later a grade.

In [8]:
# Define one attempt: copy the desk, let the agent work, then read back the records it left.
from cbnb.agent import trace_documents
from cbnb.couchbase_io import ensure_collection, upsert_docs
from datetime import datetime

SUITE = datetime.now().strftime("%m%d-%H%M%S")        # tells this run's stored attempts from any other
llm = LLM(temperature=TEMPERATURE)
SYSTEM = ("You are a support agent for a small online shop. Customers write to you by email. "
          "Use the tools to look things up and to act on the customer's behalf. "
          "Every customer contact is logged as a ticket, and the shop's policy pages say how to "
          "handle each situation. When you are finished, reply to the customer in a few sentences.")
traces_collection = ensure_collection(cluster, BUCKET, SCOPE, "traces")
grades_collection = ensure_collection(cluster, BUCKET, SCOPE, "grades")


def attempt(task, phase, k=0, review=None):
    arm = "checked" if review else "plain"
    run_id = f"{SUITE}-{phase}-{arm}-{task.id}-{k}"
    backend = desk.Desk(cluster, BUCKET, SCOPE, run_id)
    backend.prepare()                                       # identical clean state, private to this attempt
    trace = tooluse.run(
        task.id, task.message, llm=llm, system=SYSTEM, tools=desk.TOOLS, execute=backend.call,
        writes=desk.WRITES, review=(lambda: review(backend)) if review else None,
        temperature=TEMPERATURE, max_turns=MAX_TURNS, run_id=run_id,
        suite=SUITE, phase=phase, arm=arm, k=k)
    upsert_docs(traces_collection, trace_documents([trace]), progress=False)
    docs = backend.documents()                              # the terminal state, read back from Couchbase
    return {"task": task, "phase": phase, "arm": arm, "k": k, "trace": trace, "docs": docs,
            "effects": desk.diff(SEED, docs)}

First the shape of a run, so there is something to grade. The task is `return-posted`: a customer
has posted a return and wants the tracking number added to their ticket. Policy says to add it
as a note and put the ticket on hold until the parcel arrives. The agent works, then tells the
customer it is done.

In [9]:
# Run the agent once on the return-posted task, and show every tool call and its final message.
def show_trace(trace):
    for step in trace.steps:
        arguments = ", ".join(f"{k}={v!r}" for k, v in step.action["arguments"].items())
        outcome = "ok" if step.observation["ok"] else "ERROR: " + step.observation["error"]
        print(f"{step.n:>2}. {step.action['name']}({arguments[:110]}) -> {outcome}")
    print("\nFinal message:", trace.meta["final_message"])


task = BY_ID["return-posted"]
one = attempt(task, "one")
print(task.message, "\n")
show_trace(one["trace"])

Dana Whitfield (dana.whitfield@mail.example). I posted the shirt back today, tracking number 1ZX99A4400. Can you add that to my return? 

 1. get_customer(email='dana.whitfield@mail.example') -> ok
 2. find_tickets(customer_id='C-109') -> ok
 3. add_note(text='Customer shipped the shirt back on 2026-10-02; tracking number 1ZX99A4400.', ticket_id='T-9002') -> ok

Final message: Done! I've added tracking number 1ZX99A4400 to your return ticket (T-9002) for the shirt from order O-5010. Once we receive it, we'll process your size L exchange. Thanks for letting us know!


That reads like a job done. The way most agent evaluations would check it is the **trajectory**:
the attempt finished with a message rather than running out of turns, it called a tool that
changes something, and its last tool call did not return an error. This is the definition the
article uses.

In [10]:
# Judge the attempt the way a trajectory check does: did it finish, change something, and end without an error.
from cbnb.readout import Item, Panel


def proxy(trace):
    finished = trace.stopped == "satisfied"
    changed = any(step.action["name"] in desk.WRITES for step in trace.steps)
    clean = bool(trace.steps) and trace.steps[-1].observation["ok"]
    return {"passed": finished and changed and clean, "finished": finished, "changed": changed, "clean": clean}


def proxy_panel(verdict):
    return Panel(
        "Trajectory check: passed" if verdict["passed"] else "Trajectory check: failed",
        [("What it looks at", [
            Item("ok" if verdict["finished"] else "blocked", "finished with a final message"),
            Item("ok" if verdict["changed"] else "blocked", "called a tool that changes records"),
            Item("ok" if verdict["clean"] else "blocked", "last tool call returned no error")])],
        status="ok" if verdict["passed"] else "blocked")


proxy_panel(proxy(one["trace"]))

Trajectory check: passed

What it looks at
  [ok] finished with a final message
  [ok] called a tool that changes records
  [ok] last tool call returned no error

The records say whether the work was done. `desk.diff` compares the attempt's final documents
with the seed and lists what changed. `grade` then does two things with that list. It checks each
record the task requires: the record has to exist, or the attempt has a **missing required
effect**, and each of its fields has to hold the right value, or the attempt has a **wrong field
value**. And it looks at what else changed: anything not on the task's `may_change` list is an
**unintended extra effect**. These are the article's three labels, and one attempt can have more
than one.

In [11]:
# Grade an attempt on the records it left: required fields first, then anything that changed that should not have.
def grade(task, docs, effects):
    checks, problems = [], []
    for expect in task.expects:
        doc = expect.find(docs)
        if doc is None:
            checks.append({"ok": False, "what": expect.what, "detail": "no such record was left"})
            problems.append({"label": "missing required effect", "where": expect.what})
            continue
        for name, want in expect.fields.items():
            got = doc.get(name)
            ok = want(got) if callable(want) else got == want
            checks.append({"ok": bool(ok), "what": f"{expect.what}: {name}",
                           "detail": f"{got!r}" if ok else f"is {got!r}, needs {want!r}"})
            if not ok:
                problems.append({"label": "wrong field value", "where": expect.what, "field": name,
                                 "expected": repr(want), "got": got})
    allowed = Counter(task.may_change)
    for effect in effects:
        pattern = effect["key"] if effect["op"] != "created" else f"{effect['collection']}/new"
        if allowed[pattern] > 0:
            allowed[pattern] -= 1
        else:
            checks.append({"ok": False, "what": f"{effect['op']} {effect['key']}",
                           "detail": "nothing on this task should have changed that"})
            problems.append({"label": "unintended extra effect", "where": effect["key"], "op": effect["op"]})
    return {"task_id": task.id, "passed": not problems, "problems": problems, "checks": checks,
            "labels": sorted({p["label"] for p in problems})}


def state_panel(result):
    wrong = [c for c in result["checks"] if not c["ok"]]
    return Panel(
        "Final records: correct" if result["passed"] else f"Final records: {len(wrong)} problem{'s' * (len(wrong) != 1)}",
        [("Checked against the task", [
            Item("ok" if c["ok"] else "blocked", c["what"], c["detail"]) for c in result["checks"]])],
        status="ok" if result["passed"] else "blocked")


def judge(record):
    # Grade an attempt both ways and store the verdict beside its trace.
    record["proxy"] = proxy(record["trace"])
    record["grade"] = grade(record["task"], record["docs"], record["effects"])
    trace, verdict, result = record["trace"], record["proxy"], record["grade"]
    upsert_docs(grades_collection, {f"grade::{trace.run_id}": {
        "type": "grades", "suite": SUITE, "phase": record["phase"], "arm": record["arm"], "k": record["k"],
        "run_id": trace.run_id, "task_id": record["task"].id, "passed": result["passed"],
        "proxy_passed": verdict["passed"], "labels": result["labels"], "problems": result["problems"],
        "effects": [{"key": e["key"], "op": e["op"]} for e in record["effects"]],
        "model_calls": trace.meta["model_calls"],
        "tokens": trace.meta["prompt_tokens"] + trace.meta["completion_tokens"]}}, progress=False)
    return record


one = judge(one)
print(desk.describe(one["effects"]))
state_panel(one["grade"])

changed tickets/T-9002: notes: [{"on": "2026-10-02", "text": "Customer wants to return it for a size L."}] -> [{"on": "2026-10-02", "text": "Customer wants to return it for a size L."}, {"on": "2026-10-04", "text": "Customer shipped the shirt back on 2026-10-02; tracking number 1ZX99A4400."}]


Final records: 1 problem

Checked against the task
  [--] the shirt-return ticket: status  is 'open', needs 'hold'
  [ok] the shirt-return ticket: notes  [{'on': '2026-10-02', 'text': 'Customer wants to return it for a size L.'}, {'on': '2026-10-04', 'text': 'Customer shipped the shirt back on 2026-10-02; tracking number 1ZX99A4400.'}]

The two checks look at the same attempt. If they disagree on your run, the agent finished
cleanly and told the customer the work was done while the records say otherwise: nothing in the
trace is wrong, and one field is. If they agree, this task went well this time. The next section
shows how often the two disagree across all ten tasks, which one run cannot.

## 4. Every task once

Now each task once, with the trajectory verdict and the state verdict side by side. The pool runs
a few attempts at a time, as `K` attempts at ten tasks would take a long while one by one, and
the provider limits how many requests may arrive together.

In [12]:
# Run every task once, concurrently, from clean state.
from concurrent.futures import ThreadPoolExecutor


def run_all(jobs):
    with ThreadPoolExecutor(WORKERS) as pool:
        return list(pool.map(lambda job: judge(attempt(*job[:2], **job[2])), jobs))


once = run_all([(task, "once", {"k": 0}) for task in TASKS])
once_table = pd.DataFrame([{
    "task": r["task"].id,
    "trajectory check": "pass" if r["proxy"]["passed"] else "fail",
    "final records": "right" if r["grade"]["passed"] else "wrong",
    "what was wrong": ", ".join(r["grade"]["labels"]) or ""} for r in once])
once_table

,task,trajectory check,final records,what was wrong
0,carrier-exception,pass,right,
1,past-the-window,pass,right,
2,ticket-already-open,pass,wrong,wrong field value
3,one-item-damaged,pass,right,
4,refund-already-issued,pass,right,
5,order-number-wrong,fail,wrong,missing required effect
6,late-gold-customer,pass,right,
7,lost-in-transit,pass,right,
8,return-posted,pass,wrong,wrong field value
9,order-in-transit,fail,wrong,missing required effect


In [13]:
# Count the attempts the trajectory check passed and the final records failed.
trajectory_passed = sum(r["proxy"]["passed"] for r in once)
records_right = sum(r["grade"]["passed"] for r in once)
fooled = sum(r["proxy"]["passed"] and not r["grade"]["passed"] for r in once)
print(f"Trajectory check passed {trajectory_passed} of {len(once)} attempts; the final records were right for {records_right}.")
print(f"{fooled} attempts passed the trajectory check and left wrong records.")

Trajectory check passed 8 of 10 attempts; the final records were right for 6.
2 attempts passed the trajectory check and left wrong records.


Any row with `pass` in the first column and `wrong` in the second is an attempt a trajectory
check would have counted as a success. A row can also fail both ways. On `order-in-transit` the
agent often answers the customer's question and never opens a ticket, so there is no
state-changing call for the trajectory check to find and no ticket for the records to hold. The
next section runs everything `K` times.

## 5. Every task `K` times

One attempt is one sample from a model that is sampling. Here every task runs `K` times, each
from its own clean copy of the desk, and the results are summarised with the article's three
numbers:

- **pass@1**: the share of all attempts that left the right records.
- **pass@k**: the share of tasks the agent got right at least once in `K` tries. This is what it
  can do.
- **observed k/k**: the share of tasks it got right every time. This is what it can be relied on
  to do.

In [14]:
# Run every task K times from clean state, and turn the results into one row per attempt.
def runs_table(results):
    return pd.DataFrame([{
        "task": r["task"].id, "arm": r["arm"], "k": r["k"],
        "passed": r["grade"]["passed"], "proxy": r["proxy"]["passed"],
        "labels": r["grade"]["labels"],
        "model calls": r["trace"].meta["model_calls"],
        "tool calls": len(r["trace"].steps),
        "tokens": r["trace"].meta["prompt_tokens"] + r["trace"].meta["completion_tokens"]}
        for r in results])


plain = run_all([(task, "repeat", {"k": k}) for task in TASKS for k in range(K)])
plain_runs = runs_table(plain)
len(plain_runs)

100

In [15]:
# Compute pass@1, pass@k and observed k/k, and the share of attempts a trajectory check would have passed.
def reliability(runs):
    per_task = runs.groupby("task").agg(passes=("passed", "sum"), tries=("passed", "size"))
    failed = runs[~runs.passed]
    return {
        "attempts": len(runs),
        "pass@1": runs.passed.mean(),
        "pass@k": (per_task.passes >= 1).mean(),
        "k/k": (per_task.passes == per_task.tries).mean(),
        "trajectory check passes": runs.proxy.mean(),
        "failed attempts": len(failed),
        "failed but passed the trajectory check": int(failed.proxy.sum()),
        "model calls per attempt": runs["model calls"].mean(),
        "tokens per attempt": runs.tokens.mean()}


plain_m = reliability(plain_runs)
pd.Series({
    f"pass@1 (share of the {plain_m['attempts']} attempts that left the right records)": f"{plain_m['pass@1']:.0%}",
    f"pass@{K} (share of tasks passed at least once)": f"{plain_m['pass@k']:.0%}",
    f"observed {K}/{K} (share of tasks passed every time)": f"{plain_m['k/k']:.0%}",
    "trajectory check would have passed": f"{plain_m['trajectory check passes']:.0%} of attempts"}).to_frame("agent, no extra check")

,"agent, no extra check"
pass@1 (share of the 100 attempts that left the right records),62%
pass@10 (share of tasks passed at least once),90%
observed 10/10 (share of tasks passed every time),40%
trajectory check would have passed,83% of attempts


In [16]:
# Show, for each task, how many of its K attempts left the right records, and how many a trajectory check would have passed.
per_task = plain_runs.groupby("task", sort=False).agg(
    **{f"right records (of {K})": ("passed", "sum"), f"trajectory check passed (of {K})": ("proxy", "sum")})
per_task["trajectory passed, records wrong"] = plain_runs[plain_runs.proxy & ~plain_runs.passed].groupby("task").size()
per_task.fillna(0).astype(int)

,right records (of 10),trajectory check passed (of 10),"trajectory passed, records wrong"
task,,,
carrier-exception,10,10,0
past-the-window,9,9,0
ticket-already-open,2,10,8
one-item-damaged,10,10,0
refund-already-issued,9,9,0
order-number-wrong,1,5,4
late-gold-customer,10,10,0
lost-in-transit,10,10,0
return-posted,1,10,9


pass@1 counts every attempt equally, so it hides how the passes are spread. The table shows
the spread. The tasks fall into three groups: some pass every time, some almost never pass, and
some pass part of the time. The ones that pass every time tend to be tasks where the policy page
the agent finds says exactly what to do. The ones that rarely pass tend to be tasks where the
right ending includes a step nothing prompts: a status to change, or a ticket to open.

The gap between pass@k and observed k/k is the share of tasks whose single result you could not
have trusted. A task the agent passed once is a task it *can* do. A task it passed ten times in
ten is one you could hand it. On this desk those are different lists, and a benchmark that ran
each task once would have reported the first as if it were the second.

Read the last column of the per-task table next to the first two. Where the trajectory check
passed an attempt and the records were wrong, the agent had done everything a log would show.
Those are the attempts a trajectory check cannot catch, however it is tuned.

## 6. What went wrong, queried from Couchbase

Each attempt left three kinds of document: its final desk records, a trace of every step, and a
grade. Because they are documents in a collection, the questions a reader would ask of a run
are queries. These are real SQL++ statements over this run's attempts, and they take little
change to run on your own agent's traces. `UNNEST` opens an array held inside each document,
here the list of problems a grade found and the list of steps a trace took.

In [17]:
# Define a helper that runs a SQL++ statement over this run's stored attempts and returns a table.
from couchbase.n1ql import QueryScanConsistency
from couchbase.options import QueryOptions

GRADES, TRACES = f"`{BUCKET}`.`{SCOPE}`.grades", f"`{BUCKET}`.`{SCOPE}`.traces"


def sql(statement, **params):
    options = QueryOptions(named_parameters={"suite": SUITE, **params},
                           scan_consistency=QueryScanConsistency.REQUEST_PLUS)
    return pd.DataFrame(list(cluster.query(statement, options)))

**How did the failed attempts fail?** Each failed attempt is counted once under every label it
earned, so the counts can add up to more than the number of failures.

In [18]:
# Count the failed attempts under each of the three failure labels.
sql(f'''
    SELECT label, COUNT(*) AS attempts
    FROM {GRADES} g UNNEST g.labels AS label
    WHERE g.suite = $suite AND g.phase = "repeat" AND g.arm = "plain" AND g.passed = false
    GROUP BY label ORDER BY attempts DESC''')

,attempts,label
0,21,wrong field value
1,17,missing required effect


**Which wrong values did the agent leave behind?** The fields that held a wrong value, and what
they held instead.

In [19]:
# List the wrong field values left most often, with the task and what the field should have held.
sql(f'''
    SELECT g.task_id AS task, p.field AS `field`, p.expected AS needed, TO_STRING(p.got) AS found,
           COUNT(*) AS attempts
    FROM {GRADES} g UNNEST g.problems p
    WHERE g.suite = $suite AND g.phase = "repeat" AND g.arm = "plain"
      AND p.label = "wrong field value"
    GROUP BY g.task_id, p.field, p.expected, TO_STRING(p.got)
    ORDER BY attempts DESC LIMIT 10''').style.set_properties(**{"text-align": "left"})

,attempts,field,found,needed,task
0,9,status,open,'hold',return-posted
1,8,status,open,'solved',ticket-already-open
2,4,status,open,'solved',order-number-wrong


**Which failures would a trajectory check have passed?** These are the attempts that finished
cleanly, changed something, hit no error at the end, and still left the wrong records.

In [20]:
# Count, per task, the failed attempts that a trajectory check would have passed.
sql(f'''
    SELECT g.task_id AS task, COUNT(*) AS attempts,
           SUM(CASE WHEN g.proxy_passed THEN 1 ELSE 0 END) AS passed_the_trajectory_check
    FROM {GRADES} g
    WHERE g.suite = $suite AND g.phase = "repeat" AND g.arm = "plain" AND g.passed = false
    GROUP BY g.task_id ORDER BY passed_the_trajectory_check DESC, attempts DESC''')

,attempts,passed_the_trajectory_check,task
0,9,9,return-posted
1,8,8,ticket-already-open
2,9,4,order-number-wrong
3,10,0,order-in-transit
4,1,0,refund-already-issued
5,1,0,past-the-window


**Where did tool handling go wrong?** The article's third point is that most failures are failures
to handle a tool's response. The traces hold every step, so a query can ask which tools returned
an error, what the error said, and how many of those errors came in attempts that went on to
fail.

In [21]:
# List the tool errors the agent met, by tool, and how many came in attempts whose records ended up wrong.
sql(f'''
    SELECT s.action.name AS tool, SUBSTR(s.observation.error, 0, 70) AS error,
           COUNT(*) AS times,
           SUM(CASE WHEN t.run_id IN failed THEN 1 ELSE 0 END) AS in_failed_attempts
    FROM {TRACES} t UNNEST t.steps s
    LET failed = (SELECT RAW g.run_id FROM {GRADES} g
                  WHERE g.suite = $suite AND g.phase = "repeat" AND g.arm = "plain" AND g.passed = false)
    WHERE t.suite = $suite AND t.phase = "repeat" AND t.arm = "plain" AND s.observation.ok = false
    GROUP BY s.action.name, SUBSTR(s.observation.error, 0, 70)
    ORDER BY times DESC LIMIT 8''').style.set_properties(**{"text-align": "left"})

,error,in_failed_attempts,times,tool
0,"invalid status 'on hold'; a ticket's status is one of open, hold, solv",0,6,update_ticket
1,"invalid status 'on_hold'; a ticket's status is one of open, hold, solv",0,1,update_ticket


Two of the three labels account for them all. In a **wrong field value**, the right record exists
and one field is wrong, and nearly always that field is a ticket's `status`, left `open` where
policy says `hold` or `solved`. A **missing required effect** is a record the agent never wrote,
most often the ticket itself. The third label, an
unintended extra effect, would be a refund policy forbids or a duplicate ticket, and none of
these failures was one.

The last query explains why a trajectory check cannot see these. The only tool errors the agent
met were invalid ticket statuses, such as `on hold` for `hold`, and the error text names the
valid ones. The agent recovered from every one of them, and none came in a failed attempt. The
attempts that failed never hit an error to recover from. They stopped one step short, with
nothing to tell them so.

The article's third point is that most failures are tool handling. That is a result about its
tools and its tasks. On this desk the failures are a different shape: judgement that no tool
response contradicts. The two are not in conflict, but they call for different fixes, and the
way to tell which one you have is to query the traces.

## 7. The fix: read back what you changed before you say you are done

The article suggests that an agent check the terminal state before it commits, not trust its own
summary of it. Here that is one extra step in the loop. When the agent first says it is
finished, the loop does not accept it. It reads the agent's own documents back from Couchbase,
works out what changed against the seed, and shows the agent that list. The agent may confirm
and finish, or make corrections with the same tools.

The agent never sees what the task required or any of the checks. All it learns is what it
actually changed, and it still has to judge whether that is right. After it finishes, the
records are graded exactly as before.

The same ten tasks run `K` times each, so the two arms can be compared on all three numbers.
The first arm is the section above, run with the same code. This one costs more, and the
comparison reports that next to any gain.

In [22]:
# Define the review: show the agent every change its own attempt has made to the records.
def review_message(backend):
    return ("Before you finish, here is every change your actions have made to the shop's records "
            "so far:\n\n" + desk.describe(backend.effects()) + "\n\nIf these changes are what the "
            "situation needs, reply to the customer as before. If anything is wrong or missing, fix "
            "it with the tools first.")


print(review_message(desk.Desk(cluster, BUCKET, SCOPE, one["trace"].run_id)))

Before you finish, here is every change your actions have made to the shop's records so far:

changed tickets/T-9002: notes: [{"on": "2026-10-02", "text": "Customer wants to return it for a size L."}] -> [{"on": "2026-10-02", "text": "Customer wants to return it for a size L."}, {"on": "2026-10-04", "text": "Customer shipped the shirt back on 2026-10-02; tracking number 1ZX99A4400."}]

If these changes are what the situation needs, reply to the customer as before. If anything is wrong or missing, fix it with the tools first.


In [23]:
# Run every task K times again, with the review step before the agent is allowed to finish.
checked = run_all([(task, "repeat", {"k": k, "review": review_message}) for task in TASKS for k in range(K)])
checked_runs = runs_table(checked)
checked_m = reliability(checked_runs)
len(checked_runs)

100

In [24]:
# Put the two arms side by side on pass@1, pass@k, observed k/k, and what each cost.
compare = pd.DataFrame({"agent, no extra check": plain_m, "agent with the review step": checked_m})
shown = compare.copy().astype(object)
for row in ["pass@1", "pass@k", "k/k", "trajectory check passes"]:
    shown.loc[row] = [f"{v:.0%}" for v in compare.loc[row]]
for row in ["attempts", "failed attempts", "failed but passed the trajectory check"]:
    shown.loc[row] = [f"{v:.0f}" for v in compare.loc[row]]
shown.loc["model calls per attempt"] = [f"{v:.1f}" for v in compare.loc["model calls per attempt"]]
shown.loc["tokens per attempt"] = [f"{v:,.0f}" for v in compare.loc["tokens per attempt"]]
shown.rename(index={"k/k": f"observed {K}/{K}", "pass@k": f"pass@{K}"})

,"agent, no extra check",agent with the review step
attempts,100,100
pass@1,62%,68%
pass@10,90%,90%
observed 10/10,40%,60%
trajectory check passes,83%,100%
failed attempts,38,32
failed but passed the trajectory check,21,32
model calls per attempt,5.0,6.4
tokens per attempt,"7,797","10,842"


In [25]:
# Show how many of each task's K attempts left the right records, with and without the review step.
by_task = pd.concat({"no extra check": plain_runs.groupby("task", sort=False).passed.sum(),
                     "with the review step": checked_runs.groupby("task", sort=False).passed.sum()}, axis=1)
by_task.index.name = f"right records, of {K}"
by_task

,no extra check,with the review step
"right records, of 10",,
carrier-exception,10,10
past-the-window,9,10
ticket-already-open,2,3
one-item-damaged,10,10
refund-already-issued,9,10
order-number-wrong,1,0
late-gold-customer,10,10
lost-in-transit,10,10
return-posted,1,1


Showing the agent its own changes before it finished helped a little: pass@1 went up by a few
points and observed k/k did not go down. It cost about a third more model calls and 40% more
tokens per attempt. Those are the numbers to weigh against each other, and with ten tasks a
single task moving from sometimes to always is ten points of observed k/k.

Two details in the table are worth more than the totals. The trajectory check passes almost every
attempt in the review arm, so it is even less able than before to tell the good attempts from
the bad ones. And the gain is uneven. A diff shows what changed, not what should have changed.
When nothing has changed at all, the diff says so, and that is hard to ignore: `order-in-transit`
is the task where the agent most often answers and never opens a ticket, and it is where the
review step helps most. When something changed but one field is still wrong, such as a ticket
left `open`, the diff has nothing to object to, and tasks built around that kind of omission,
such as `return-posted` and `ticket-already-open`, tend to move little.

## What to take from this

- **A trajectory is a weak check.** Every attempt in which the agent finished, changed a record
  and hit no final error counts as a pass by the trajectory definition, and at least half of the
  failed attempts here were among them. Whatever the agent got wrong, it got wrong quietly.
- **One run is one sample.** The same agent passed some tasks every time and some only
  sometimes. The share of tasks it passed at least once is well above the share it passed every
  time, so a single run flatters it, and the size of the flattery depends on which tasks you
  happened to run.
- **Keeping every attempt's records makes both checks cheap.** The records are documents, so
  grading is a diff plus a few conditions, and the questions that follow, which wrong values,
  which failures a proxy would have missed, which tool errors came before a failure, are
  SQL++ over collections you already have.
- **Check the check.** Reading back what the agent changed before accepting its "done" helped a
  little here and cost real tokens. It helped most where nothing at all had been
  done, and little where one field had been left wrong. Measure it
  on your own tasks before relying on it, and measure it the same way as the agent: more than
  once.

This is one model on a small invented desk, with tasks that were written to have a tempting wrong
ending and no simulated user. The comparisons are the part to trust. The absolute rates belong to
this desk.

## Where to take this

- **A simulated user.** Each task here is one message. Many support failures need a second
  turn: the agent asks a question, the customer answers, and the right ending depends on the
  answer. The benchmark has dialogue, and a model playing the customer would add it.
- **More models.** This is one model. The article's point that similar pass@1 can hide very
  different k/k can only be checked across several, with the same tasks and the same `K`.
- **Stage the writes in a transaction.** The review step above shows the agent what it has
  already changed. Couchbase transactions allow a stricter version: make the agent's writes
  inside a transaction, run the check against the uncommitted state, and commit only if it
  passes. A wrong ending then never reaches the records at all.
- **Run a few real ThinkingBox tasks.** The benchmark is open. Its tasks, state checks and
  tool servers would give numbers that mean something against other agents, which this desk's
  cannot.

## What produced these results

Every number above came from the model below. A different model, or the same name served
differently, will give different numbers.

In [26]:
# Name the models that produced the results above.
cbnb.run_card()

,model,how it ran
Chat model,z-ai/glm-5.3,"via NanoGPT; 1,204 calls"


In [27]:
# Optional clean-up: delete the desk, every attempt's records, the traces and the grades. Uncomment to run.
# drop_demo_data(cluster, BUCKET, SCOPE)